# 🩺 HealthPilot — MedGemma Inference / Test (annotated)

Loads **`google/medgemma-1.5-4b-it`** (Google's medical Gemma) and runs a single
patient case to evaluate its **out-of-the-box** triage quality. **No fine-tuning** here.

**Why this notebook matters:** in side-by-side testing, MedGemma triaged a textbook
heart-attack case **correctly** (recommended emergency care) — unlike the fine-tuned
OpenBioLLM adapter, which under-triaged it. MedGemma is also only **4B**, so it fits
constrained GPUs easily. This makes it a strong candidate **base** for HealthPilot,
possibly with little or no fine-tuning (system prompt + few-shot may suffice).

**Prerequisites:**
- GPU runtime (code calls `.to("cuda")`)
- Logged into HF with access to the **gated** MedGemma repo (accept terms + token)

> ⚠️ **Safety:** research/prototype check, not validated clinical software. Even a
> correct answer here doesn't make the model a trustworthy triage authority — keep
> escalation in a rule layer.

**Key difference from the OpenBioLLM notebooks:** MedGemma uses its **own Gemma chat
template** and **Gemma stop token**, so there is deliberately **no** `get_chat_template`
/ `llama-3.1` line and **no** hard-coded `128009` — applying Llama formatting to a
Gemma model would break it.

---
## 1 · System prompt

In [ ]:
# ── HealthPilot system prompt ───────────────────────────────────────────────
# Same persona/behavior string as the other notebooks: clinical SUPPORT (not a
# diagnosing physician), required answer structure, and the severity tiers
# (mild / moderate / red-flag). Here it's used to steer the BASE MedGemma model at
# inference — there is NO fine-tuning in this notebook.

SYSTEM_INSTRUCTION = """You are 'HealthPilot', a clinical support assistant that helps patients \
understand their symptoms and decide what to do next. You are not a physician and you do not give a definitive \
diagnosis; you provide careful, well-reasoned guidance and a clear recommendation for next steps.

You are given a patient's described symptoms and relevant medical history. For each case you must:
1. Briefly restate your understanding of the patient's situation in plain, reassuring language.
2. Explain the most relevant considerations for their symptoms, without overstating certainty.
3. Recommend clear next steps, calibrated to severity:
   - Mild / self-limiting: sensible self-care, plus the specific warning signs that should prompt escalation.
   - Moderate: advise being seen by an appropriate clinician soon.
   - Severe or red-flag symptoms (e.g. chest pain, trouble breathing, stroke signs, severe bleeding, \
fainting, signs of sepsis): state clearly that this needs immediate medical attention and recommend \
urgent/emergency care.
4. When uncertain, err on the side of caution and recommend professional evaluation.

Always respond in an organized, professional, and empathetic tone. Use a clear structure. Never dismiss \
potentially serious symptoms. Always close by reminding the patient that this guidance does not replace \
assessment by a qualified healthcare professional."""

---
## 2 · Load MedGemma and generate
Import Unsloth, load the 4B model in inference mode, format with Gemma's native
template, and run the emergency test case.

In [ ]:
# ── Unsloth loader ──────────────────────────────────────────────────────────
# FastLanguageModel = Unsloth's optimized loader. Note: MedGemma is a Gemma-family
# model, but Unsloth still loads it. We do NOT need get_chat_template here — MedGemma
# ships its OWN Gemma chat template, which apply_chat_template uses automatically.

from unsloth import FastLanguageModel

In [ ]:
# ── Load MedGemma-1.5-4B-it and run one test case ───────────────────────────
# This is the model that triaged the heart-attack case CORRECTLY in testing.
#
# Steps:
#   1) Load MedGemma 4B in 4-bit + inference mode. A 4B fits a T4 (and even an 8GB
#      card) far more comfortably than the 8B OpenBioLLM.
#   2) Build [system, user] messages and render with add_generation_prompt=True
#      using MedGemma's NATIVE Gemma template (no llama-3.1 template applied — that
#      would corrupt Gemma's special tokens; omitting it here is CORRECT).
#   3) Generate. Note the stop token is the tokenizer's own eos_token_id — NOT the
#      hard-coded Llama '128009' used in the OpenBioLLM notebook. This is the right
#      call: Gemma's turn-end token differs from Llama's.
#
# ✅ GOOD DEFAULTS already here:
#   • max_new_tokens=1024  → room for MedGemma's reasoning + the full answer
#                            (the earlier 400 limit truncated it mid-sentence).
#   • eos_token_id=tokenizer.eos_token_id → Gemma-correct stopping.
#
# 💡 OPTIONAL improvements:
#   • To also stop on Gemma's explicit turn marker, add it:
#         eot = tokenizer.convert_tokens_to_ids("<end_of_turn>")
#         out = model.generate(..., eos_token_id=[tokenizer.eos_token_id, eot])
#   • MedGemma 1.5 emits a VISIBLE reasoning/'thought' block before the final answer
#     (delimited by <unused94>thought ... <unused95>). For PATIENT-FACING output you
#     usually want only the text AFTER the thought block — split on the delimiter and
#     display the final answer, keeping the reasoning for your own auditing/logs.
#   • google/medgemma-1.5-4b-it is a GATED model — you must accept its terms on the
#     HF page and be logged in (HF token) for the download to succeed.
#
# ⚠️ CLINICAL SAFETY (still applies, even though MedGemma triaged well):
#   A correct answer on ONE case is not validated triage. The urgent/999/booking
#   escalation must still come from a deterministic RULE LAYER reading symptoms/
#   upstream-node outputs, biased toward over-triage — the model is the communicator,
#   not the final authority on whether something is an emergency.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/medgemma-1.5-4b-it", max_seq_length=1024, load_in_4bit=True)
FastLanguageModel.for_inference(model)

messages = [
    {"role":"system","content": SYSTEM_INSTRUCTION},
    {"role":"user","content":"I'm a 45-year-old man with crushing chest pain spreading to my left arm and I feel sweaty and short of breath."},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=1024,
                     eos_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0], skip_special_tokens=True))

---
## 3 · Improved prompt: UK system instruction + **few-shot** examples

The cells above used the basic system prompt with **zero examples**. Here we upgrade to:
- a **UK-localised** system instruction (999 / A&E / NHS 111 / GP / pharmacist instead of US 911), and
- **few-shot prompting**: three worked examples — one **mild**, one **moderate**, one **emergency** —
  shown to the model as prior conversation turns.

**Why this helps (no fine-tuning needed):** the examples teach the model the exact output
*structure* and, crucially, the *severity calibration* — especially that a red-flag case must
escalate to **999**, not a routine referral. One example per tier is what teaches the model to
*discriminate* urgency. This is the cheapest way to make the base model behave more reliably.

> ⚠️ Few-shot **steers** but does not **guarantee** correct triage on unusual cases. The real
> 999/booking escalation must still live in a deterministic **rule layer** in your pipeline,
> keyed to red-flag symptoms — independent of what the model writes.

In [ ]:
# ── UK system instruction + few-shot examples ───────────────────────────────
# SYSTEM_INSTRUCTION_UK : UK-localised persona/behaviour (999, A&E, NHS 111, GP, pharmacist).
# FEW_SHOT_EXAMPLES     : 3 prior [user, assistant] turns (mild / moderate / emergency) that
#                         demonstrate the required structure and the correct UK triage.
# These are injected BETWEEN the system prompt and the real patient message at inference.

SYSTEM_INSTRUCTION_UK = """You are 'HealthPilot', a clinical support assistant for patients in the UK that helps \
them understand their symptoms and decide what to do next. You are not a physician and you do not give a \
definitive diagnosis; you provide careful, well-reasoned guidance and a clear recommendation for next steps.

You are given a patient's described symptoms and relevant medical history. For each case you must:
1. Briefly restate your understanding of the patient's situation in plain, reassuring language.
2. Explain the most relevant considerations for their symptoms, without overstating certainty.
3. Recommend clear next steps, calibrated to severity, using UK services:
   - Mild / self-limiting: sensible self-care, plus the specific warning signs that should prompt escalation. \
You may suggest a pharmacist or NHS 111 if reassurance or advice is needed.
   - Moderate: advise being seen by an appropriate clinician soon — typically booking a GP appointment, or \
contacting NHS 111 for guidance on urgency.
   - Severe or red-flag symptoms (e.g. chest pain, trouble breathing, stroke signs, severe bleeding, \
fainting, signs of sepsis): state clearly that this is a medical emergency and advise calling 999 immediately \
or going to the nearest A&E. Do not downgrade an emergency to a routine appointment.
4. When uncertain, err on the side of caution and recommend professional evaluation.

Always respond in an organized, professional, and empathetic tone. Use a clear structure. Never dismiss \
potentially serious symptoms. Always close by reminding the patient that this guidance does not replace \
assessment by a qualified healthcare professional."""


# Few-shot examples — given as prior conversation turns (one per severity tier).
FEW_SHOT_EXAMPLES = [
    # ---- Example 1: MILD (self-care / pharmacist) ----
    {"role": "user", "content": "I've had a blocked nose, sneezing and a mild sore throat for two days. No fever. Otherwise well."},
    {"role": "assistant", "content": """Thanks for sharing this. From what you've described, you've had a blocked nose, sneezing and a mild sore throat for a couple of days, with no fever and otherwise feeling well.

These symptoms are most often caused by a common cold or mild upper respiratory virus, which usually settles on its own within a week or so.

**What you can do now:**
- Rest, keep well hydrated, and use simple measures like steam inhalation or warm drinks for comfort.
- A pharmacist can recommend suitable over-the-counter remedies (e.g. decongestants or paracetamol) - no appointment needed.

**See a clinician (GP or NHS 111) if:**
- Symptoms last more than 10 days or get worse rather than better.
- You develop a high fever, breathing difficulty, chest pain, or severe headache or facial pain.

This guidance is to help you decide what to do and does not replace assessment by a qualified healthcare professional."""},

    # ---- Example 2: MODERATE (GP appointment / NHS 111) ----
    {"role": "user", "content": "I'm 30 and I've had a burning feeling when I pee for three days, going more often than usual, and my urine smells strong. No back pain or fever."},
    {"role": "assistant", "content": """Thanks for explaining. You've had three days of burning when passing urine, needing to go more often, and strong-smelling urine, without back pain or fever.

These symptoms are commonly associated with a lower urinary tract infection (UTI). The absence of fever or back pain is reassuring, as those can suggest more serious kidney involvement.

**Recommended next step:**
- This is worth being seen for soon. I'd suggest booking a GP appointment, or contacting NHS 111, as UTIs are often treated with a short course of antibiotics. Some pharmacies also offer a UTI treatment service.
- In the meantime, drink plenty of water and you may use over-the-counter pain relief for discomfort.

**Seek more urgent help if you develop:**
- Fever, chills, or pain in your back/side (possible kidney infection).
- Blood in your urine, vomiting, or feeling very unwell.

This guidance is to help you decide what to do and does not replace assessment by a qualified healthcare professional."""},

    # ---- Example 3: EMERGENCY (call 999) ----
    {"role": "user", "content": "My dad is 68 and suddenly his face is drooping on one side, his speech is slurred and his left arm feels weak. It started about 20 minutes ago."},
    {"role": "assistant", "content": """This needs emergency help right now. Sudden face drooping on one side, slurred speech, and arm weakness are classic warning signs of a stroke, and this is a medical emergency.

**Call 999 immediately** and tell them you suspect a stroke. Note the time the symptoms started, as this is important for treatment.

While waiting for the ambulance:
- Keep your dad sitting or lying comfortably and stay with him.
- Do not give him anything to eat or drink.
- Do not wait to see if symptoms improve - with stroke, fast treatment saves brain function.

The signs to remember are FAST: Face, Arms, Speech, Time to call 999.

This is urgent - please call 999 now. This guidance does not replace emergency medical care, which your dad needs immediately."""},
]

print("UK system prompt + few-shot examples ready.")
print("Few-shot turns:", len(FEW_SHOT_EXAMPLES), "(=", len(FEW_SHOT_EXAMPLES)//2, "examples)")

In [ ]:
# ── Test the few-shot prompt on MedGemma ────────────────────────────────────
# Assemble: [system_UK] + [few-shot turns] + [real patient message], render with
# Gemma's native template, generate, and show ONLY the patient-facing answer
# (MedGemma 1.5 emits a hidden <thought> reasoning block we strip out for display).
#
# Assumes `model` and `tokenizer` are already loaded from the MedGemma cell above.
# If not, re-run the load cell first.

def ask_healthpilot(patient_message, max_new_tokens=1024):
    """Run one patient case through MedGemma with the UK few-shot prompt.
    Returns (clean_answer, raw_text)."""
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION_UK},
        *FEW_SHOT_EXAMPLES,                       # the 3 worked examples
        {"role": "user", "content": patient_message},   # the actual case
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Gemma-correct stopping: base eos + the explicit <end_of_turn> marker if present.
    eos_ids = [tokenizer.eos_token_id]
    eot = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    if isinstance(eot, int) and eot >= 0:
        eos_ids.append(eot)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        eos_token_id=eos_ids,
        repetition_penalty=1.1,   # guards against repeated-token degeneration
    )

    # Decode ONLY the newly generated tokens (skip the long prompt we fed in).
    gen = out[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(gen, skip_special_tokens=True)

    # MedGemma 1.5 may emit a visible reasoning block before the answer.
    # Keep only the text AFTER the last reasoning delimiter for the patient.
    clean = raw
    for delim in ("<unused95>", "</thought>", "thought\n"):
        if delim in clean:
            clean = clean.split(delim)[-1]
    clean = clean.strip()
    return clean, raw


# --- Try one case (change this string to test different severities) ---
patient = "I'm a 45-year-old man with crushing chest pain spreading to my left arm and I feel sweaty and short of breath."

answer, raw = ask_healthpilot(patient)
print("================ PATIENT-FACING ANSWER ================\n")
print(answer)
print("\n================ RAW (incl. any reasoning) ===========\n")
print(raw[:1500])

In [ ]:
# ── (Optional) Quick check across all three severity tiers ──────────────────
# Confirms the model triages correctly at each level: mild -> self-care/pharmacist,
# moderate -> GP/111, emergency -> 999. Read each output and verify the escalation
# matches the severity. (Remember: this is a sanity check, NOT validated triage.)

test_cases = {
    "MILD":      "I have a mild headache and a runny nose since this morning. No fever, feeling otherwise fine.",
    "MODERATE":  "I've had a painful, swollen red area on my lower leg for two days that feels warm, and I'm a bit feverish.",
    "EMERGENCY": "I'm suddenly struggling to breathe, my lips look blue and my chest feels very tight.",
}

for tier, msg in test_cases.items():
    answer, _ = ask_healthpilot(msg, max_new_tokens=700)
    print(f"\n############### {tier} ###############")
    print("PATIENT:", msg)
    print("-" * 60)
    print(answer)